# E001 — Replay Data Factory

**Input checklist**
- Required: V2 suite.
- Preferred: attached replay JSON dataset or a directory of downloaded `episode-*-replay.json` files.
- Optional: Episodes Index catalog from E000.
- Accelerator: **None / CPU**.
- Internet: **OFF** if replay JSONs are attached; **ON** only if you choose to use the Kaggle CLI to fetch public episodes.

**Outputs:** `turns.parquet` and `daily_macros.parquet`. Opponent private inventory is never used as an online feature; when present it may be used only as an offline training target.

In [ ]:
from pathlib import Path
import os,sys,json
SUITE_CANDIDATES=[Path('/kaggle/input/kaggriculture-v2-suite'),Path('/kaggle/input/kaggriculture-v2-suite/kaggriculture_v2_suite'),Path.cwd().parent,Path.cwd()]
ROOT=next((p for p in SUITE_CANDIDATES if (p/'src'/'kagv2').exists()),None)
if ROOT is None: raise FileNotFoundError('Attach/upload kaggriculture_v2_suite as a Kaggle Dataset, or run this notebook inside the repo.')
sys.path.insert(0,str(ROOT)); WORK=Path('/kaggle/working/kagv2') if Path('/kaggle/working').exists() else ROOT/'artifacts'; WORK.mkdir(parents=True,exist_ok=True)
print('ROOT=',ROOT,'WORK=',WORK)

### Replay acquisition
The official Kaggle CLI supports `competitions team-submissions`, `competitions episodes`, and `competitions replay`. Prioritize **recent current-engine episodes** from highly rated submissions plus a diversity sample, rather than scraping everything indiscriminately.

In [ ]:
import subprocess,glob,pandas as pd
SUBMISSION_IDS=[]
DOWNLOAD_WITH_CLI=False
REPLAY_OUT=WORK/'replays';REPLAY_OUT.mkdir(exist_ok=True)
if DOWNLOAD_WITH_CLI:
    for sid in SUBMISSION_IDS:
        r=subprocess.run(['kaggle','competitions','episodes',str(sid),'-v'],capture_output=True,text=True)
        print(r.stdout[:2000]); print(r.stderr[:500])


In [ ]:
roots=[Path('/kaggle/input'),REPLAY_OUT,ROOT/'replays'] if Path('/kaggle/input').exists() else [REPLAY_OUT,ROOT/'replays']
paths=[]
for r in roots:
    if r.exists(): paths.extend([p for p in r.rglob('*.json') if 'replay' in p.name.lower() or 'episode' in p.name.lower()])
paths=sorted(set(paths));print('replays found:',len(paths));print(*paths[:10],sep='\n')
if not paths: raise RuntimeError('No replay JSONs found. Download public episodes with the Kaggle CLI or attach them as a Kaggle Dataset.')

In [ ]:
from src.kagv2.replay import paths_to_turn_frame,add_outcome_labels,add_future_opponent_sell_labels
from src.kagv2.features import daily_macro_frame
turns=paths_to_turn_frame(paths,stride=1)
turns=add_outcome_labels(turns);turns=add_future_opponent_sell_labels(turns,horizon=24)
print(turns.shape);display(turns.head())
turns.to_parquet(WORK/'turns.parquet',index=False)
daily=daily_macro_frame(turns);daily.to_parquet(WORK/'daily_macros.parquet',index=False)
print('daily',daily.shape);display(daily.head())

In [ ]:
assert turns.groupby(['episode_id','player']).size().min()>0
assert not any(c.startswith('opp_shed_') for c in [x for x in turns.columns if x.startswith('price_')])
print('episodes',turns.episode_id.nunique(),'players',turns[['episode_id','player']].drop_duplicates().shape[0])
print('final reward median',turns.groupby(['episode_id','player']).final_reward.last().median())